<a href="https://colab.research.google.com/github/lulusbersamasama-TE/solar-ai-mppt-optimization/blob/main/notebooks/03_integrated_mppt_po_simulation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 03 — P&O vs AI-MPPT integration

Load the trained models from notebooks 01 and 02, compare their commanded voltage to a numeric P&O tracker using **held-out P–V curves**, and separately test synthetic shading ratios linked to a held-out GHI forecast. Metrics are based on **equal controller iterations**, not seconds or actual joules. AI assumes access to six section irradiances that P&O does not have.

In [ ]:
from google.colab import drive
from pathlib import Path
import os

# Menghubungkan Google Drive
drive.mount('/content/drive')

# Folder proyek di Google Drive
project = Path(
    '/content/drive/MyDrive/solar_ai_mppt_project'
)

# Memastikan folder tersedia
(project / 'data').mkdir(parents=True, exist_ok=True)
(project / 'outputs').mkdir(parents=True, exist_ok=True)

# Menyesuaikan lokasi dengan notebook
link = Path('/content/solar_ai_mppt_project')

if not link.exists():
    os.symlink(
        project,
        link,
        target_is_directory=True
    )

# Memvalidasi file dataset
required = [
    'srrl_weather_compact.csv.gz',
    'pv_database_summary_extract.csv',
    'pv_iv_curves_0p1V.npz'
]

for filename in required:
    path = project / 'data' / filename
    print(filename, 'OK' if path.exists() else 'MISSING')


In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from sklearn.neighbors import NearestNeighbors
ROOT=next((p for p in [Path.cwd(),Path.cwd().parent,Path('/content/solar_ai_mppt_project')]
           if (p/'data'/'pv_database_summary_extract.csv').exists()),None)
if ROOT is None: raise FileNotFoundError('Read data/README.md; place the prepared files in data/')
DATA=ROOT/'data';OUT=ROOT/'outputs';OUT.mkdir(exist_ok=True)
SEED=42;FAST=os.getenv('SOLAR_FAST','0')=='1'

## A. Load models, holdout splits and original PV current sweeps at 0.1 V resolution

In [ ]:
needed=[OUT/'gmpp_models.joblib',OUT/'gmpp_splits.csv',
        OUT/'irradiance_test_forecasts.csv.gz',DATA/'pv_iv_curves_0p1V.npz']
missing=[str(p) for p in needed if not p.exists()]
if missing: raise FileNotFoundError('Run notebooks 01 and 02; missing: '+', '.join(missing))
models=joblib.load(OUT/'gmpp_models.joblib')
rf=models['random_forest'];linear=models['linear'];FEATURES=models['feature_columns']
meta=pd.read_csv(DATA/'pv_database_summary_extract.csv')
meta['curve_id']=meta.Curve.astype(int)
meta=meta.merge(pd.read_csv(OUT/'gmpp_splits.csv'),on='curve_id',validate='one_to_one')
meta=meta.set_index('curve_id',drop=False)
matrix=np.load(DATA/'pv_iv_curves_0p1V.npz')
V=matrix['voltage'].astype(float)
I=matrix['current'].astype(float)
ids=matrix['curve_id'].astype(int)
assert I.shape==(14400,415) and np.array_equal(ids,np.arange(1,14401))
P=I*V[None,:]
ideal_p=P.max(axis=1);ideal_v=V[P.argmax(axis=1)]
err=np.abs(ideal_p-meta.loc[ids,'Pmax_W'].to_numpy())
print('Max power-grid deviation (W)',err.max(),'median',np.median(err))
test_meta=meta.loc[meta.split=='test'].copy()
def power_at(cid,v):
    v=float(np.clip(v,V.min(),V.max()))
    return float(max(0,v*np.interp(v,V,I[int(cid)-1])))
def curve_p(cid): return V*I[int(cid)-1]

## B. Common simulated controller-step limit and P&O rule

P&O observes current power and reverses direction if it falls below the previous sample. AI moves toward a predicted voltage, respecting the same 1-V maximum step. Each curve is a **static episode**; time-domain converter dynamics are not modeled.

In [ ]:
def step_to(v,target,max_step=1):
    return float(np.clip(v+np.clip(target-v,-max_step,max_step),V[0],V[-1]))
def episode(cid,targets,n_steps=30,start_v=27.0,vstep=1.0):
    cid=int(cid); vpo=float(start_v); direction=1.0; previous=None
    vai={name:float(start_v) for name in targets}
    records=[]
    for k in range(n_steps):
        ppo=power_at(cid,vpo)
        row={'step':k,'curve_id':cid,'po_v':vpo,'po_power':ppo,
             'ideal_v':ideal_v[cid-1],'ideal_power':ideal_p[cid-1]}
        for label,target in targets.items():
            vai[label]=step_to(vai[label],target,vstep)
            row[label+'_v']=vai[label];row[label+'_power']=power_at(cid,vai[label])
        records.append(row)
        if previous is not None and ppo<previous: direction=-direction
        previous=ppo
        vpo=float(np.clip(vpo+vstep*direction,V[0],V[-1]))
    return pd.DataFrame(records)
def summarize(log,names):
    last=log.groupby('event',sort=False).tail(1)
    rows=[]
    for name in names:
        pk='po_power' if name=='po' else name+'_power'
        vk='po_v' if name=='po' else name+'_v'
        reached=log.loc[log[pk]>=.98*log.ideal_power]
        first=reached.groupby('event').step.min()
        last10=log.groupby('event',sort=False).tail(10)
        rows.append({'controller':name,
             'tracking_efficiency_pct':100*log[pk].sum()/log.ideal_power.sum(),
             'last_step_GMPP_hit_pct':100*np.mean(last[pk]>=.98*last.ideal_power),
             'mean_power_W':log[pk].mean(),
             'first_hit_iter_median':float(first.median()) if len(first) else np.nan,
             'mean_last10_voltage_pp_V':last10.groupby('event')[vk].apply(lambda z:z.max()-z.min()).mean(),
             'mean_last10_power_std_W':last10.groupby('event')[pk].std(ddof=0).fillna(0).mean()})
    return pd.DataFrame(rows)

## C. Test A — held-out static partial-shading curves

Every algorithm begins at 27 V and runs 30 voltage-update iterations per episode. Static curves are simulated separately, not asserted to represent physical switching transients.

In [ ]:
sample=test_meta.sample(n=min(len(test_meta),25 if FAST else 120),random_state=SEED)
logs=[]
for event,(_,r) in enumerate(sample.iterrows()):
    x=r[FEATURES].to_numpy(float).reshape(1,-1)
    refs={'ai_rf':float(np.clip(rf.predict(x)[0],V[0],V[-1])),
          'ai_linear':float(np.clip(linear.predict(x)[0],V[0],V[-1]))}
    s=episode(r.curve_id,refs)
    s['event']=event
    logs.append(s)
log_a=pd.concat(logs,ignore_index=True)
metrics_a=summarize(log_a,['po','ai_linear','ai_rf'])
print(metrics_a.round(3).to_string(index=False))
log_a.to_csv(OUT/'static_mppt_log.csv',index=False)
metrics_a.to_csv(OUT/'static_mppt_metrics.csv',index=False)
cid=int(sample.iloc[0].curve_id)
fig,ax=plt.subplots(figsize=(9,4))
ax.plot(V,curve_p(cid),label='Recorded PV sweep')
ax.axvline(ideal_v[cid-1],ls=':',color='black',label='Global MPP')
for name,model in [('Linear',linear),('RF',rf)]:
    x=sample.iloc[0][FEATURES].to_numpy(float).reshape(1,-1)
    pred=float(model.predict(x)[0])
    ax.scatter([pred],[power_at(cid,pred)],label=name)
ax.set(xlabel='Voltage (V)',ylabel='Power (W)',title=f'Unseen curve {cid}')
ax.legend();plt.tight_layout();fig.savefig(OUT/'pv_curve_example.png',dpi=150);plt.show()

## D. Test B — synthetic shading + weather forecasting

From **held-out** SRRL GHI, assume fixed shading ratios for six sections, use actual future GHI to select the nearest *held-out* PV sweep, and compare P&O to AI with persistence vs AI with RF forecast. This is a conditional synthetic scenario, not an observed dynamic PV plant. The oracle uses actual future GHI only for an upper-bound diagnostic.

In [ ]:
weather=pd.read_csv(OUT/'irradiance_test_forecasts.csv.gz',
                    parse_dates=['timestamp','target_timestamp'])
weather=weather.loc[weather.ghi_now.between(850,1050)&
    weather.ghi_true_5min.between(850,1050)&
    weather.ghi_pred_rf.between(850,1050)&
    weather.ghi_pred_persistence.between(850,1050)].copy()
if weather.empty: raise RuntimeError('No overlap with PV irradiance range: report out of domain')
count=min(len(weather),15 if FAST else 75)
snap=weather.iloc[np.linspace(0,len(weather)-1,count).astype(int)].reset_index(drop=True)
pool=test_meta.loc[(test_meta[FEATURES].min(axis=1)/test_meta[FEATURES].max(axis=1))<.6]
if pool.empty: pool=test_meta
templates=pool.sample(n=min(8,len(pool)),random_state=SEED+10)
ratios=templates[FEATURES].to_numpy(float)
ratios=ratios/ratios[:,:1]
nn=NearestNeighbors(n_neighbors=1).fit(test_meta[FEATURES].to_numpy(float))
test_ids=test_meta.curve_id.to_numpy(int)
logs=[];matches=[]
for event,r in snap.iterrows():
    shade=ratios[event%len(ratios)]
    future=shade*float(r.ghi_true_5min)
    pred=shade*float(r.ghi_pred_rf)
    persist=shade*float(r.ghi_now)
    distance,index=nn.kneighbors(future.reshape(1,-1))
    matching_rmse=float(distance[0,0]/np.sqrt(len(FEATURES)))
    if matching_rmse>80: continue
    cid=int(test_ids[index[0,0]])
    refs={'ai_forecast':float(np.clip(rf.predict(pred.reshape(1,-1))[0],V[0],V[-1])),
          'ai_persist':float(np.clip(rf.predict(persist.reshape(1,-1))[0],V[0],V[-1])),
          'ai_oracle':float(np.clip(rf.predict(future.reshape(1,-1))[0],V[0],V[-1]))}
    s=episode(cid,refs);s['event']=event;logs.append(s)
    matches.append({'event':event,'curve_id':cid,'timestamp':r.timestamp,
                    'target_timestamp':r.target_timestamp,'matching_RMSE_Wm2':matching_rmse,
                    'ghi_actual_future':r.ghi_true_5min,'ghi_pred_future':r.ghi_pred_rf})
if not logs: raise RuntimeError('No well-matched held-out PV curves for the synthetic scenarios')
log_b=pd.concat(logs,ignore_index=True);match_df=pd.DataFrame(matches)
metrics_b=summarize(log_b,['po','ai_persist','ai_forecast','ai_oracle'])
print('Accepted forecast-linked episodes:',len(matches),'/',len(snap))
print('Median matching error (W/m² per section)',match_df.matching_RMSE_Wm2.median())
print(metrics_b.round(3).to_string(index=False))
log_b.to_csv(OUT/'forecast_integrated_mppt_log.csv',index=False)
metrics_b.to_csv(OUT/'forecast_integrated_mppt_metrics.csv',index=False)
match_df.to_csv(OUT/'forecast_scenario_matches.csv',index=False)
fig,ax=plt.subplots(figsize=(9,4))
ax.bar(metrics_b.controller,metrics_b.tracking_efficiency_pct)
ax.set(ylabel='Numerical tracking efficiency (%)',title='Synthetic forecast-linked episodes')
ax.set_ylim(0,105);plt.xticks(rotation=25);plt.tight_layout()
fig.savefig(OUT/'forecast_integrated_efficiency.png',dpi=150);plt.show()

## E. Scientific verification

1. Source I–V curves are sampled data, not `P=ηGA` synthesis; the 0.1-V resampling causes finite grid error.
2. Last-step ≥98% ideal power is a per-episode hit metric, not verified hardware settling time.
3. The forecast-linked test simulates a **hypothetical** fixed section shading ratio and uses nearest **test-only** curve matching. Matching errors and out-of-domain failures must be reported.
4. AI is privileged with six section irradiance estimates; basic P&O lacks equivalent sensing.
5. When forecast and persistence produce similar MPPT efficiency, do **not** claim forecast-based energy benefit; the future-GHI oracle is not a practical controller.
6. Actual converter dynamics, switching losses and sensor costs are not modeled.